# 07 — Sensitivity analysis

**Deliverable D15.**

Every headline number rests on choices forced by the data rather than chosen on merit.
This sweeps them so each result ships with a **range** rather than a point estimate.

| axis | why it is in doubt |
|---|---|
| `reactive_orientation` | unresolved — 213 sites fit as-delivered, 106 flipped |
| capacity basis | no nameplate; the whole curve is scaled by an *observed* quantile |
| `voltage_aggregation` | `mean` (correct for three-phase) vs `max` (legacy Method A) |
| `tolerance_fraction` | ±4% re-anchored to `s_99` because there is no nameplate |
| night-anomaly sites | 5 likely storage + 15 stray timestamps, in or out |
| peak-hour window | the legacy query used an inclusive `BETWEEN` |

**How to read a sweep.** A number that barely moves is robust to that choice. A number
that moves a lot is *conditional* on it and must be reported as such — never averaged
across the sweep, which would invent a value no defensible configuration produces.

This is the slow notebook: each row is a full interval-level rescore.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents)
                  if (p / "oem_analysis").is_dir() and (p / "bms_sa_review").is_dir()), None)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from oem_analysis.config import se_config as C
from oem_analysis.lib import se_store, se_contract as contract, se_params
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
con = se_store.connect()
config, params = se_params.CONFIG, se_params.PARAMS
from oem_analysis.lib import se_sensitivity as sens
from oem_analysis.lib import se_conformance as cf
from oem_analysis.lib import se_plots as plots
from oem_analysis.lib import se_adverse as adv

adverse = adv.classify_adverse_sites(con, config)
display(pd.DataFrame([{"axis": k, "changes": str(v) or "(defaults)"}
                      for k, v in sens.DEFAULT_SWEEPS.items()]))

## Volt-VAr conformance sweep

Watch `assessable_intervals` alongside the rate. A sweep that moves the rate by moving
the **denominator** is telling you something quite different from one that moves the
numerator.

In [ ]:
conf_sweep = sens.sweep_conformance(con, config=config, params=params)
display(conf_sweep)

In [ ]:
display(sens.tornado(conf_sweep, "reduced_nonconf_pct"))
display(sens.tornado(conf_sweep, "pct_sites_conformant"))

## Method A sweep

Watch `energy_kWh` against `symptom_intervals`. The capacity basis moves both, in
**opposite directions**: a lower `s_limit` makes the apparent-limit test fire more often
(raising the count) while shrinking the headroom displacement per interval (lowering the
energy per event). Reporting only one of them would mislead.

In [ ]:
a_sweep = sens.sweep_method_a(con, config=config, params=params, adverse=adverse)
display(a_sweep)

In [ ]:
display(sens.tornado(a_sweep, "energy_kWh"))

conf_sweep.to_csv(C.ARTEFACT_DIR / "sensitivity_conformance.csv", index=False)
a_sweep.to_csv(C.ARTEFACT_DIR / "sensitivity_method_a.csv", index=False)
print(f"-> {C.ARTEFACT_DIR / 'sensitivity_conformance.csv'}")
print(f"-> {C.ARTEFACT_DIR / 'sensitivity_method_a.csv'}")

## Minimum-interval sensitivity

A site with **one** exposed interval can only score 0% or 100%. The 10% rule has no
meaning there — the verdict is decided by a single reading — yet such a site currently
carries the same weight in the fleet rate as one observed for a thousand intervals.

**What the reference implementations do**, checked directly:

- **Hossein's original** applies *no* minimum, for either response. The only count filter
  in `SolA2024_Analysis` is `total_count > 5` in `sustained_operation.ipynb` — a different
  analysis. There is no site-level Volt-Watt rate in that repo at all.
- **`bms_sa_review`** sets `min_site_intervals = 1` (i.e. none, matching the original) but
  provides `minimum_interval_sensitivity` and **plots it for both Volt-Watt variants** in
  `02_conformance_curtailment_analysis.ipynb`.

So the defensible position is not a chosen threshold — it is this curve, reported. The
sweep below keeps `min = 1` as the baseline and runs to 20.

In [ ]:
vvar_site_day = cf.voltvar_site_day(con, config, params)
vwatt_site_day = cf.voltwatt_site_day(con, config)

print("How thin is the evidence? — Volt-Watt")
display(sens.min_interval_exposure_profile(vwatt_site_day, "voltwatt", config))

### Volt-Watt

`pct_dropped_conformant` is the column that settles the interpretation. It is the share of
the **discarded** sites that were conformant:

- near **50%** → the filter is neutral, and any movement in the headline is real;
- well **above 50%** → the filter is removing well-behaved sparse sites, so raising the
  minimum drives the rate **down**, and the fall is an artefact of the rule rather than
  worse inverters.

In [ ]:
vw_min_sweep = sens.sweep_min_intervals(vwatt_site_day, "voltwatt", config)
display(vw_min_sweep.drop(columns="denominator"))
display(plots.plot_min_interval_sweep(
    vw_min_sweep, "Volt-Watt — minimum exposed intervals (V > 253 V)"))

### Volt-VAr

Same sweep against the capability-assessable denominator. Expect it to be far flatter:
every site in this fleet has assessable Volt-VAr intervals and the median count is orders
of magnitude larger, so the rule has almost nothing to bite on. That contrast is the point
— it shows the sensitivity is a property of **Volt-Watt exposure**, not of the 10% rule.

In [ ]:
vv_min_sweep = sens.sweep_min_intervals(vvar_site_day, "voltvar", config)
display(vv_min_sweep.drop(columns="denominator"))
display(plots.plot_min_interval_sweep(
    vv_min_sweep, "Volt-VAr — minimum capability-assessable intervals"))

### Reading it

Compare the two curves before choosing anything. If Volt-Watt moves materially and
Volt-VAr does not, the minimum-interval rule is not a global analysis choice — it is
specific to how rarely this fleet sees 253 V, and it should be reported as such rather
than applied silently to both.

Whatever is chosen, `min_intervals` is already a parameter on
`voltwatt_verdict_measures`, `site_verdict_measures` and `denominator_comparison`, so
adopting a threshold is a one-line change in notebook 03 — not a rebuild.

In [ ]:
summary = pd.DataFrame({
    "min_intervals": vw_min_sweep.min_intervals,
    "voltwatt_pct_conformant": vw_min_sweep.pct_conformant,
    "voltwatt_pct_sites_dropped": vw_min_sweep.pct_sites_dropped,
    "voltvar_pct_conformant": vv_min_sweep.pct_conformant,
    "voltvar_pct_sites_dropped": vv_min_sweep.pct_sites_dropped,
})
display(summary)

vw_span = vw_min_sweep.pct_conformant.max() - vw_min_sweep.pct_conformant.min()
vv_span = vv_min_sweep.pct_conformant.max() - vv_min_sweep.pct_conformant.min()
print(f"Headline moves across min=1..20:  Volt-Watt {vw_span:.1f} pp   "
      f"Volt-VAr {vv_span:.1f} pp")

vw_min_sweep.to_csv(C.ARTEFACT_DIR / "min_interval_sweep_voltwatt.csv", index=False)
vv_min_sweep.to_csv(C.ARTEFACT_DIR / "min_interval_sweep_voltvar.csv", index=False)

## What this establishes

The tornado tables name the choices each headline number is conditional on. Anything
near the top belongs in the limitations section, not a footnote.

On the current fleet the largest mover for Method A energy is the **capacity basis**
(`s_95` raises it ~67%), followed by the **reactive orientation** (~−28%). Both are
consequences of the same two gaps: no nameplate, and an unresolved sign convention.
Neither is a modelling preference, and both should be presented as data limitations.